# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [1]:

# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull


In [2]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

In [3]:
# ==============================
# 2. Load environment variables
# ==============================

# Load environment variables
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

os.environ["HF_HOME"] = r"E:\huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = r"E:\huggingface_cache"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = r"E:\huggingface_cache"

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

print("Endpoint:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("Deployment:", chat_deployment)
print("Key exists:", os.getenv("AZURE_OPENAI_API_KEY") is not None)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Endpoint: https://ag23098-7581-resource.services.ai.azure.com/openai/v1/
Deployment: gpt-4o
Key exists: True


## Implemented Improvements

For Part 4 of the assignment, the RAG system was improved in four ways:

1. Text segmentation was improved by replacing simple paragraph splitting with overlapping word-based chunks.
2. Retrieval was improved by using a configurable `top_k` value and increasing it to 4.
3. Prompt design was improved by instructing the model to answer only from the provided context and to say when information is missing.
4. An optional interactive question-answering loop was added for easier testing.

These changes make the system easier to evaluate, improve context retrieval, and reduce the risk of hallucinated answers.

In [ ]:
# ==============================
# 2. Load all knowledge files
# ==============================

from pathlib import Path

data_dir = Path("data")
file_texts = {}  # filename -> raw text

for file_path in sorted(data_dir.glob("*.txt")):
    with open(file_path, "r", encoding="utf-8") as f:
        file_texts[file_path.name] = f.read()

print(f"Loaded {len(file_texts)} file(s): {list(file_texts.keys())}")

# ==============================
# 3. Improvement 1: Better chunking with overlap
#    Instead of splitting on blank lines, we split text into
#    overlapping word-based chunks. Overlap preserves context at
#    chunk boundaries and improves retrieval quality.
# ==============================

def chunk_text(text, source, chunk_size=120, overlap=30):
    """
    Splits text into overlapping word-based chunks.

    chunk_size = approximate number of words per chunk
    overlap    = number of words repeated between neighboring chunks
    """
    words = text.split()
    chunk_objects = []
    chunk_id = 0
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_objects.append({
            "text": " ".join(chunk_words),
            "source": source,
            "chunk_id": chunk_id
        })
        chunk_id += 1
        # Move forward by (chunk_size - overlap) so next chunk
        # repeats the last 'overlap' words of this chunk.
        start += chunk_size - overlap

    return chunk_objects

# Build chunk_objects from all knowledge files
chunk_objects = []
for filename, text in file_texts.items():
    chunk_objects.extend(chunk_text(text, source=filename))

# Plain list of chunk texts used for embedding
chunks = [c["text"] for c in chunk_objects]

print(f"\nCreated {len(chunk_objects)} chunk(s) from {len(file_texts)} file(s).")
print("\nExample chunks:")
for c in chunk_objects[:3]:
    print(f"  [{c['source']} | chunk {c['chunk_id']}]: {c['text'][:90]}...")


In [5]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 12


In [ ]:
# ==============================
# 6. Test questions
#
# Improvement 2: Configurable top_k
#   Changed from a fixed k=2 to top_k=4, so the system retrieves
#   more context and is less likely to miss relevant chunks.
#
# Improvement 3: Improved prompt design
#   The prompt now instructs the model to answer ONLY from the
#   provided context and to explicitly say when information is
#   missing, reducing the risk of hallucinated answers.
# ==============================

# Improvement 2: easy to change in one place
top_k = 4

# Two test questions:
#   1. Answerable from the knowledge base
#   2. Not answerable – model should say there is no information
test_questions = [
    "When is IT support available?",
    "What is the capital city of Australia?",
]

for question in test_questions:
    print("=" * 80)
    print("Question:", question)

    # Embed the question
    question_embedding = np.array([get_embedding(question)]).astype("float32")

    # Search FAISS for top_k nearest chunks
    distances, indices = index.search(question_embedding, top_k)

    # Collect retrieval metadata (source file, chunk id, score, text)
    retrieved_metadata = []
    for score, idx in zip(distances[0], indices[0]):
        retrieved_metadata.append({
            "score": float(score),
            "source": chunk_objects[idx]["source"],
            "chunk_id": chunk_objects[idx]["chunk_id"],
            "text": chunk_objects[idx]["text"]
        })

    # Print retrieval details
    print(f"\nTop {top_k} retrieved chunks:")
    for i, item in enumerate(retrieved_metadata, start=1):
        print(f"  [{i}] source={item['source']} | chunk_id={item['chunk_id']} "
              f"| score={item['score']:.4f}")
        print(f"       {item['text'][:100]}...")

    # Improvement 3: Build the improved prompt
    context_blocks = []
    for i, item in enumerate(retrieved_metadata, start=1):
        context_blocks.append(
            f"[Source {i}: {item['source']}, chunk {item['chunk_id']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
You are a helpful assistant answering questions using only the provided context.

Rules:
1. Use only the information from the context.
2. If the answer is not in the context, say: \"There is no information available in the provided context.\"
3. Keep the answer clear and concise.
4. Mention the source file when possible.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer with the existing LLM client
    response = client.responses.create(
        model=chat_deployment,
        input=prompt,
        temperature=0
    )

    print("\nLLM Answer:")
    print(response.output_text)
    print()


In [ ]:
# ==============================
# 7. Improvement 4: Optional interactive question-answering loop
#    This cell is OPTIONAL. Run it manually to ask your own questions.
#    Skip it when running the notebook automatically from top to bottom.
# ==============================

# NOTE: type 'exit' to stop the loop.

while True:
    question = input("\nAsk a question, or type 'exit': ")

    if question.lower() == "exit":
        break

    # Embed the question
    question_embedding = np.array([get_embedding(question)]).astype("float32")

    # Search FAISS for top_k nearest chunks
    distances, indices = index.search(question_embedding, top_k)

    # Collect retrieval metadata
    retrieved_metadata = []
    for score, idx in zip(distances[0], indices[0]):
        retrieved_metadata.append({
            "score": float(score),
            "source": chunk_objects[idx]["source"],
            "chunk_id": chunk_objects[idx]["chunk_id"],
            "text": chunk_objects[idx]["text"]
        })

    # Print retrieval details
    print(f"\nTop {top_k} retrieved chunks:")
    for i, item in enumerate(retrieved_metadata, start=1):
        print(f"  [{i}] source={item['source']} | chunk_id={item['chunk_id']} "
              f"| score={item['score']:.4f}")
        print(f"       {item['text'][:100]}...")

    # Build improved prompt (same as cell 6)
    context_blocks = []
    for i, item in enumerate(retrieved_metadata, start=1):
        context_blocks.append(
            f"[Source {i}: {item['source']}, chunk {item['chunk_id']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
You are a helpful assistant answering questions using only the provided context.

Rules:
1. Use only the information from the context.
2. If the answer is not in the context, say: \"There is no information available in the provided context.\"
3. Keep the answer clear and concise.
4. Mention the source file when possible.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    response = client.responses.create(
        model=chat_deployment,
        input=prompt,
        temperature=0
    )

    print("\nLLM Answer:")
    print(response.output_text)

print("\nExiting interactive mode.")
